In [1]:
import torch
import importlib
import numpy as np
from scipy.optimize import minimize
from modele import *
from visualisation import *
import pytorch_optim
importlib.reload(pytorch_optim)
from pytorch_optim import *
%load_ext autoreload
%autoreload 2


In [2]:
N=3
nb_colonnes_default = 50
T = 300
L = 1
P0 = torch.tensor([[0, 1 / 2, 1], [1 / 2, 0, 0], [1 / 2, 1 / 2, 0]]) 
params = {
    "N": N,  # nombre de couches
    "L": L,  # longueur du bassin
    "H": 1,  # hauteur du bassin
    "nb_colonnes_default": 50,  # discrétisation horizontale
    "D": 1e-5,  # coefficient de diffusion
    "T": T,  # temps final pour 20 tours
    "CFL": 0.95,  # facteur CFL
    "u" : [0.1]*N,
    "I_s": 1500, #1500 initialement
    "epsilon": 4.6, #4.6 pour une luminosité de 1% au fond du bassin
    "k_d": 2.99e-4,   # 2.99*10e-5 = 2.99e-4
    "k_r": 4.8e-4,    # 4.8*10e-5 = 4.8e-4
    "k_h": 3.64e-4,   # 3.64*10e-5 = 3.64e-4
    "tau": 6.849,
    "sigma_H": 2.9e-3,  # 2.9*10e-4 = 2.9e-3
    "theta": 3.64e-4 * 2.9e-3,  # k_h * sigma_H
    "I_star": np.sqrt(4.8e-4 / (2.99e-4 * 6.849 * (2.9e-3)**2)),
    "mu_max": 3.64e-4 * 2.9e-3 / (6.849 * 2.9e-3 + 2*np.sqrt((2.99e-4 * 6.849 * (2.9e-3)**2)/4.8e-4))
}

Objet de l'étude : trouver un motif dans le composition des matrices de mélange (de permutation) pour différentes valeurs de N. On fera ensuite le calcul pour un nombre de couches plus élevé.

In [3]:
for n in range(3, 6):
    params["N"] = n 
    params["u"] = [0.1]*n
    meilleure_masse, meilleure_matrice, _ = meilleure_matrice_de_permutation(X_ini_one_layer(params["N"], params["nb_colonnes_default"]), T, params)
    print(meilleure_masse, meilleure_matrice)

[[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
  1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
  1. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]]


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (2, 3) + inhomogeneous part.

In [ ]:
for n in range(6, 8):
    params["N"] = n 
    params["u"] = [0.1]*n
    meilleure_masse, meilleure_matrice, _ = meilleure_matrice_de_permutation(X_ini_one_layer(params["N"], params["nb_colonnes_default"]), T, params)
    print(meilleure_masse, meilleure_matrice)

50.40224092256578 [[0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1.]]


KeyboardInterrupt: 

Partir de la meilleure matrice de permutation, voir si l'on trouve une meilleure matrice bistochastique.

D'abord pour N = 3

In [ ]:
M = inverse_softmax([[1., 0., 0.],
 [0., 1., 0.],
 [0., 0., 1.]])
M = torch.tensor(M)
P, biomass_list = solve_bistochastique(X_ini_one_layer_torch(params["N"], params["nb_colonnes_default"]), T, params, 20, 1000, 0.2, True, M)

tensor([[9.9991e-01, 4.5396e-05, 4.5396e-05],
        [4.5396e-05, 9.9991e-01, 4.5396e-05],
        [4.5396e-05, 4.5396e-05, 9.9991e-01]], grad_fn=<SoftmaxBackward0>)
obj: tensor(50.2081, grad_fn=<AddBackward0>) pen: tensor(3.5527e-15, grad_fn=<SumBackward0>)
tensor([[9.9986e-01, 3.1344e-05, 3.1597e-05],
        [6.7718e-05, 9.9993e-01, 3.9355e-05],
        [6.7718e-05, 3.5959e-05, 9.9993e-01]], grad_fn=<SoftmaxBackward0>)
obj: tensor(50.2122, grad_fn=<AddBackward0>) pen: tensor(7.9038e-09, grad_fn=<SumBackward0>)
tensor([[9.9980e-01, 4.1930e-05, 4.2341e-05],
        [1.0021e-04, 9.9991e-01, 3.9414e-05],
        [1.0021e-04, 4.7961e-05, 9.9992e-01]], grad_fn=<SoftmaxBackward0>)
obj: tensor(50.2133, grad_fn=<AddBackward0>) pen: tensor(2.0401e-08, grad_fn=<SumBackward0>)
tensor([[9.9970e-01, 5.8431e-05, 5.8347e-05],
        [1.4825e-04, 9.9989e-01, 4.9766e-05],
        [1.4830e-04, 5.1876e-05, 9.9989e-01]], grad_fn=<SoftmaxBackward0>)
obj: tensor(50.2141, grad_fn=<AddBackward0>) pen: ten

In [ ]:
meilleure_matrice = [[0., 0., 0., 0., 1., 0.],
 [0., 1., 0., 0., 0., 0.],
 [0., 0., 0., 1., 0., 0.],
 [1., 0., 0., 0., 0., 0.],
 [0., 0., 1., 0., 0., 0.],
 [0., 0., 0., 0., 0., 1.]]

In [ ]:
params["N"] = 6
params["u"] = [0.1]*6
M = [[float(m) for m in l] for l in meilleure_matrice]
print(M)
M = inverse_softmax(M)
print(M)
P_bisto, biomass_list_bisto = solve_bistochastique(X_ini_one_layer_torch(params["N"], params["nb_colonnes_default"]), T, params, 50, 1000, 0.2, True, torch.tensor(M))

[[0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]
[[-5.0, -5.0, -5.0, -5.0, 0.0, -5.0], [-5.0, 0.0, -5.0, -5.0, -5.0, -5.0], [-5.0, -5.0, -5.0, 0.0, -5.0, -5.0], [0.0, -5.0, -5.0, -5.0, -5.0, -5.0], [-5.0, -5.0, 0.0, -5.0, -5.0, -5.0], [-5.0, -5.0, -5.0, -5.0, -5.0, 0.0]]
tensor([[0.0065, 0.0065, 0.0065, 0.0065, 0.9674, 0.0065],
        [0.0065, 0.9674, 0.0065, 0.0065, 0.0065, 0.0065],
        [0.0065, 0.0065, 0.0065, 0.9674, 0.0065, 0.0065],
        [0.9674, 0.0065, 0.0065, 0.0065, 0.0065, 0.0065],
        [0.0065, 0.0065, 0.9674, 0.0065, 0.0065, 0.0065],
        [0.0065, 0.0065, 0.0065, 0.0065, 0.0065, 0.9674]],
       grad_fn=<SoftmaxBackward0>)
obj: tensor(50.3888, grad_fn=<AddBackward0>) pen: tensor(0., grad_fn=<SumBackward0>)
tensor([[0.0044, 0.0096, 0.0044, 0.0044, 0.9716, 0.0096],
        [0.0044, 0.9552, 0.0044, 0.0044, 0.0044, 0.0096],